# Panzer — Gestion segura de credenciales

Este notebook explica como Panzer gestiona las API keys y secrets de forma
segura, sin que aparezcan nunca en texto plano en disco.

**Arquitectura de 3 capas:**

1. **Memoria** — cache en RAM (cifrado).
2. **Disco** — archivo `~/.panzer_creds` (valores sensibles cifrados con AES-128-CBC).
3. **Prompt** — si no existe la credencial, la pide al usuario (con `getpass` para las sensibles).

**Cifrado atado a la maquina:** la clave AES se deriva del directorio home
y la CPU del sistema. Las credenciales solo se descifran en la maquina
donde fueron creadas. No hay master password.

## 1. El cifrador AES

`AesCipher` cifra y descifra cadenas de texto. La clave se genera
automaticamente a partir de la identidad de la maquina.

In [1]:
from panzer.crypto import AesCipher

cipher = AesCipher()

# Cifrar
plaintext = "mi_api_secret_super_seguro_12345"
encrypted = cipher.encrypt(plaintext)
print(f"Texto plano:  {plaintext}")
print(f"Cifrado:      {encrypted}")

# Descifrar
decrypted = cipher.decrypt(encrypted)
print(f"Descifrado:   {decrypted}")
print(f"Coincide:     {decrypted == plaintext}")

Texto plano:  mi_api_secret_super_seguro_12345
Cifrado:      YgzGDExxCPMvTf+eaWLYwLlCf0ZxjTC355qzspoasHqBmBWnnCoQn9pzD/XdvLYY
Descifrado:   mi_api_secret_super_seguro_12345
Coincide:     True


## 2. CredentialManager

El `CredentialManager` es la interfaz de alto nivel para almacenar y
recuperar credenciales. Detecta automaticamente que nombres son sensibles.

In [2]:
from panzer.credentials import CredentialManager, _is_sensitive

# Deteccion automatica de nombres sensibles
names = ["api_key", "api_secret", "my_password", "telegram_id", "market", "base_url", "interval"]
for name in names:
    print(f"  {name:<20s}  sensible={_is_sensitive(name)}")

  api_key               sensible=True
  api_secret            sensible=True
  my_password           sensible=True
  telegram_id           sensible=True
  market                sensible=False
  base_url              sensible=False
  interval              sensible=False


### Almacenar y recuperar credenciales

Usamos un archivo temporal para esta demo (en uso real seria `~/.panzer_creds`).

In [3]:
import tempfile, os

# Crear un CredentialManager con archivo temporal (para la demo)
tmp = tempfile.mkdtemp()
cm = CredentialManager.__new__(CredentialManager)
from panzer.crypto import AesCipher
cm._cipher = AesCipher()
cm._cache = {}
cm._filepath = os.path.join(tmp, ".panzer_creds_demo")
cm._ensure_file()

# Almacenar credenciales
cm.add("api_key", "vmPUZE6mv9SD5VXXXXXXXXXXXXXXXXXXXXXXXXXXuEh8A")
cm.add("api_secret", "NhqPtmdSJXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXH5fATj0j")
cm.add("market", "spot")  # no sensible, se guarda en plano

print("Credenciales almacenadas.")

2026-03-06 12:59:36     INFO [panzer.credentials] Archivo de credenciales creado en: /tmp/tmpznfrs9pc/.panzer_creds_demo


Credenciales almacenadas.


### Que se guarda en disco?

Veamos el contenido del archivo. Las claves sensibles estan cifradas,
las no sensibles en texto plano.

In [4]:
# Contenido del archivo en disco
with open(cm.filepath) as f:
    print(f.read())

# Archivo de credenciales de Panzer
api_key = "A4Icg1SOdgRNmUm8DkbclU3ApwQlw96F2qDLkL/l2vAKx5cw2fSCXqNhfbmDc2qN"
api_secret = "XK3piQJVygNypBoMNuchhS4Pa/nh47/RnJERkGlUUdrwPKvhrGHdAQHgXaMd7c008A6cha1CWIGjoM+ROpgvf7D+LwQDeE6H3Zx6xZcJ6DQ="
market = "spot"



### Recuperar valores descifrados

In [5]:
# Sin decrypt: devuelve el valor tal como esta en cache (cifrado)
encrypted_value = cm.get("api_key", decrypt=False)
print(f"Cifrado:     {encrypted_value[:40]}...")

# Con decrypt: devuelve el valor original
decrypted_value = cm.get("api_key", decrypt=True)
print(f"Descifrado:  {decrypted_value[:40]}...")

# Las variables no sensibles se devuelven tal cual
print(f"Market:      {cm.get('market')}")

Cifrado:     A4Icg1SOdgRNmUm8DkbclU3ApwQlw96F2qDLkL/l...
Descifrado:  vmPUZE6mv9SD5VXXXXXXXXXXXXXXXXXXXXXXXXXX...
Market:      spot


### Persistencia: sobrevive al reinicio

Si limpias la cache en memoria, las credenciales se recuperan del disco.

In [6]:
# Simular reinicio: limpiar cache en memoria
cm._cache.clear()
print("Cache limpiada. Recuperando del disco...")

# Se relee automaticamente del archivo
key = cm.get("api_key", decrypt=True)
print(f"API key recuperada: {key[:20]}...")
print(f"Market recuperado:  {cm.get('market')}")

Cache limpiada. Recuperando del disco...
API key recuperada: vmPUZE6mv9SD5VXXXXXX...
Market recuperado:  spot


## 3. La firma HMAC-SHA256

`BinanceRequestSigner` usa las credenciales del `CredentialManager` para
firmar peticiones. Anade automaticamente `timestamp` y `signature` a los
parametros.

In [7]:
from panzer.exchanges.binance.signer import BinanceRequestSigner

signer = BinanceRequestSigner(credentials=cm)

# Parametros originales de una peticion
params = [("symbol", "BTCUSDT"), ("side", "BUY"), ("type", "LIMIT")]

# Firmar: anade timestamp y signature
signed = signer.sign_params(params)

print("Parametros originales:")
for k, v in params:
    print(f"  {k} = {v}")

print("\nParametros firmados:")
for k, v in signed:
    val = str(v)
    if k == "signature":
        val = val[:20] + "..."
    print(f"  {k} = {val}")

Parametros originales:
  symbol = BTCUSDT
  side = BUY
  type = LIMIT

Parametros firmados:
  symbol = BTCUSDT
  side = BUY
  type = LIMIT
  timestamp = 1772798377537
  signature = 183b791fb7a1753ad900...


### Headers con API key

Los endpoints autenticados requieren el header `X-MBX-APIKEY`.

In [8]:
headers = signer.headers_with_api_key()
print(f"Headers generados:")
for k, v in headers.items():
    print(f"  {k}: {v[:30]}...")

Headers generados:
  X-MBX-APIKEY: vmPUZE6mv9SD5VXXXXXXXXXXXXXXXX...


## 4. Uso real: primera vez vs siguientes

```
# Primera vez — no existe ~/.panzer_creds:
from panzer import BinanceClient
client = BinanceClient(market="spot")
# -> Te pide api_key y api_secret por terminal (con getpass)
# -> Los cifra y guarda en ~/.panzer_creds

# Siguientes ejecuciones:
client = BinanceClient(market="spot")
# -> Lee de ~/.panzer_creds, descifra en memoria, listo para operar
```

No necesitas gestionar credenciales manualmente. `BinanceClient` lo hace
internamente a traves de `CredentialManager` y `BinanceRequestSigner`.

### Limpiar demo

In [9]:
import shutil
shutil.rmtree(tmp)
print("Archivo temporal eliminado.")

Archivo temporal eliminado.
